# 04 — AutoCalibrate: Full Calibration Pipeline

`AutoCalibrate` runs the complete ge-transition calibration automatically:

```
res_spec → qubit_spec → power_rabi → t1 → ramsey → spin_echo → ss_opt
```

After each step it:
1. Updates `ExperimentConfig` in-place so later steps see the new values
2. Persists the result in `CalibrationStore` (timestamped JSON on disk)

You can **skip** any step with `skip=(...)` and resume just the steps you need.

In [ ]:
import sys; sys.path.insert(0, '../')
import tempfile, os

from QickworkspaceV2 import BaseExperiment, CalibrationStore, AutoCalibrate, ExperimentConfig
from QickworkspaceV2.config.system_cfg import config_list

BaseExperiment.connect_pyro4(
    ns_host='192.168.10.82', ns_port=8888, proxy_name='myqick',
    data_path=r'D:\Labber_Data\Jay\test',
)

qubit = 'Q1'
cfg_all = ExperimentConfig(config_list)
store_path = os.path.join(tempfile.gettempdir(), 'autocal_demo.json')
store = CalibrationStore(store_path, default_max_age_hours=24)


## Run the full pipeline

In [ ]:
auto = AutoCalibrate(cfg_all, qubit=qubit, cal_store=store)

# Run current pipeline steps. Bad fits can still raise RuntimeError on unconnected/invalid samples.
auto.run(skip=('spin_echo', 'ss_opt'))


## Inspect the results

In [ ]:
# AutoCalibrate.results stores calibrated parameter values keyed by config/result name.
print('=== AutoCalibrate results ===')
for key, value in auto.results.items():
    print(f'  {key:<28} {value}')


In [ ]:
# CalibrationStore — what was persisted
print('=== Calibration Store ===')
print(store.summary('Q1'))

In [ ]:
# Live config — updated in-place by AutoCalibrate
live = cfg_all.get_qubit('Q1')
print('=== Live ExperimentConfig (Q1) ===')
for key in ('res_freq_ge', 'qb_freq_ge', 'pi_gain_ge'):
    print(f'  {key:<20} = {live[key]}')

## Running only stale steps

A common pattern at the start of a lab session: check which parameters have gone stale
and only run those steps.

In [ ]:
# Map step names to the parameter they calibrate
STEP_PARAM = {
    'res_spec':   ('res_freq_ge',  48),
    'qubit_spec': ('qb_freq_ge',   12),
    'power_rabi': ('pi_gain_ge',   12),
    'ramsey':     ('qb_freq_ge',    6),
    't1':         ('T1_us',        24),
}

skip = []
run  = []
for step, (param, max_age_h) in STEP_PARAM.items():
    if store.is_stale('Q1', param, max_age_hours=max_age_h):
        run.append(step)
    else:
        skip.append(step)

print('Will run :', run)
print('Will skip:', skip)

# Then:
# auto2 = AutoCalibrate(cfg_all, 'Q1', cal_store=store)
# auto2.run(skip=tuple(skip))

## Persist to disk — reload on next session

In [ ]:
# The store auto-saves after every set().  Reload is transparent:
store2 = CalibrationStore(store_path)
print('Reloaded qb_freq_ge:', store2.get('Q1', 'qb_freq_ge'))

# Populate ExperimentConfig from the persisted store
flat = store2.to_flat_dict('Q1')
for k, v in flat.items():
    try:
        cfg_all.update(k, v, q_index='Q1')
    except Exception:
        pass   # keys like T1_us not in config are fine to skip

print('Live qb_freq_ge after reload:', cfg_all.get_qubit('Q1')['qb_freq_ge'])

**Next:** [05_custom_experiment.ipynb](05_custom_experiment.ipynb) — writing your own experiment class.